## Agent progress 代理进度

要流式传输代理进度，请使用 ` [`stream``[`astream`]`stream_mode="updates"`。这将在代理执行每个步骤后发出一个事件。例如，如果您有一个代理调用某个工具一次，您应该会看到以下更新：

- **LLM节点**：[`AIMessage`](https://reference.langchain.com/python/langchain-core/messages/ai/AIMessage)带有工具调用请求
- **工具节点**：[`ToolMessage`](https://reference.langchain.com/python/langchain-core/messages/tool/ToolMessage)执行结果
- **LLM节点**：最终AI响应

| **特性**     | **stream_mode="updates"**       | **stream_mode="messages"**     |
| ------------ | ------------------------------- | ------------------------------ |
| **比喻**     | **幻灯片演示**                  | **流动的河水**                 |
| **数据单位** | 整个节点运行后的“最终快照”。    | 每一个字符、每一个 Token。     |
| **实时性**   | 节点跑完才能看到。              | 模型刚张嘴你就能看到。         |
| **处理难度** | 简单（直接拿 `messages[-1]`）。 | 较高（需要你自己拼接字符串）。 |


In [2]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model


def get_weather(city: str) -> str:
    """Get weather for a given city."""

    return f"It's always sunny in {city}!"

model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)

agent = create_agent(
    model=model,
    tools=[get_weather],
)


### updates

> ### 什么是节点？
>
> 可以把 LangGraph 想象成一条**流水线工厂**：
>
> - **`model` 节点**：这是一个坐在中控室的**大脑**。它的任务是：看一眼现在的状态（State），然后决定是“直接回话”还是“派人去干活”。
> - **`tools` 节点**：这是一个**执行车间**。它的任务是：根据大脑发来的指令，运行具体的函数（比如 `get_weather`），然后把结果扔回流水线。
>
> 一次 `model` 思考（生成回答或决定调工具） = 运行了一次 `model` 节点。
>
> 一次 `tool` 调用（执行 Python 函数并返回结果） = 运行了一次 `tools` 节点。
>
> 当你调用 `create_agent` 时，LangGraph 在后台为你自动画了一张**环形图**：
>
> 1. **START** -> 进入 **`model` 节点**。
> 2. **`model` 节点** 运行结束：
>    - 如果它决定查天气，它会产生一个 `tool_calls`。此时 `updates` 流会报：`step: model` 结束了，它的贡献是增加了一个工具请求。
> 3. **根据连线（Edge）** -> 进入 **`tools` 节点**。
> 4. **`tools` 节点** 运行结束：
>    - 它跑完了你的 Python 函数。此时 `updates` 流会报：`step: tools` 结束了，它的贡献是增加了一条工具返回的消息。
> 5. **循环回路** -> 回到 **`model` 节点**。
> 6. **`model` 节点** 再次运行：
>    - 这次它看到天气信息了，决定直接说话。此时 `updates` 流报：`step: model` 结束了，它的贡献是最后的文本。

**`updates` 是“节点完工报告”**： 它关注的是**流程**。只有当一个节点（比如 `model` 节点）**彻底运行完**，它才会吐出这个节点对整个 `State` 做了什么修改。所以它是一整块一整块出的。

#### 1. 深度拆解：执行结果的“三步走”

在 `stream_mode="updates"` 模式下，代码会捕捉每一个节点运行后的**增量变化**。

##### **第一跳：`step: model`（决策阶段）**

- **发生了什么**：模型接收到你的问题，意识到自己不知道天气，但它发现有一个叫 `get_weather` 的工具。
- **内容**：它生成了一个 `tool_call`（工具调用请求），里面包含参数 `{'city': 'SF'}`。
- **注意**：此时模型并没说话，它只是发出了一个“去查查”的指令。

##### **第二跳：`step: tools`（执行阶段）**

- **发生了什么**：LangGraph 接收到模型的指令，自动跳转到工具节点，运行了你的 `get_weather` 函数。
- **内容**：工具返回了原始结果 `It's always sunny in SF!`。
- **注意**：这是函数跑出来的“原始素材”，还没包装成自然语言给用户。

##### **第三跳：`step: model`（总结阶段）**

- **发生了什么**：模型拿到了工具返回的“素材”，重新组织语言，给出了最终的拟人化回答。
- **内容**：`The weather in San Francisco is always sunny! 😊`。

------

#### 2. 为什么你的代码里用到了 `content_blocks`？

细心的你一定发现了，这次输出不是直接打印 `content`，而是 `content_blocks`。

这是因为在 **`version="v2"`** 的流模式下，LangChain/LangGraph 采用了更结构化的 **多模态消息块（Message Blocks）** 格式：

- **Tool Call 块**：不再是纯文本，而是一个带有 ID 和参数的结构化块。
- **Text 块**：普通的回复文本。

这种设计是为了方便前端处理——比如前端看到 `type: 'tool_call'` 就可以转圈显示“查询中”，看到 `type: 'text'` 就直接显示文字。

------

#### 3. 逻辑总结

这段代码证明了：**Agent 不是一蹴而就的，它是一个“思考 -> 行动 -> 观察 -> 再思考”的循环。**

- **`updates` 模式的价值**：它不是给你最终答案，而是给你“过程”。如果你的工具查询很慢（比如要查 5 秒钟），通过这个流，你可以让用户实时看到 Agent 正在 `step: tools` 这一步忙活，而不是让用户对着空白屏幕发呆。

In [ ]:

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="updates",
    version="v2",
):
    if chunk["type"] == "updates":
        for step, data in chunk["data"].items():
            print(f"step: {step}")
            print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'tool_call', 'id': 'f1ecdd8c-f688-4f84-9447-976073a7bfac', 'name': 'get_weather', 'args': {'city': 'SF'}}]
step: tools
content: [{'type': 'text', 'text': "It's always sunny in SF!"}]
step: model
content: [{'type': 'text', 'text': 'The weather in San Francisco is always sunny! However, for an accurate and detailed weather report, I will check the current conditions.\n'}, {'type': 'tool_call', 'id': '649a45a8-d48f-4304-bb82-fbc0d433caaa', 'name': 'get_weather', 'args': {'city': 'San Francisco'}}]
step: tools
content: [{'type': 'text', 'text': "It's always sunny in San Francisco!"}]
step: model
content: [{'type': 'text', 'text': "It seems like it's always sunny in San Francisco! For a more precise weather update, let me fetch the current conditions for you."}]


### messages


In [15]:
from typing import Generator, Dict, Any

class WebsterStreamProcessor:
    @staticmethod
    def process(stream) -> Generator[Dict[str, Any], None, None]:
        """
        解析 LangGraph v2 消息流。
        返回格式: {"type": "text", "content": "..."} 或 {"type": "tool", "name": "...", "args": "..."}
        """
        for chunk in stream:
            if chunk["type"] != "messages":
                continue
            
            for item in chunk["data"]:
                # 统一解包：无论是元组还是单个对象，都拿到 MessageChunk
                msg_chunk = item[0] if isinstance(item, (tuple, list)) else item
                
                # # --- 新增：处理思考/推理内容 ---
                # # 某些模型会把思考过程放在 reasoning_content 字段
                # if hasattr(msg_chunk, "reasoning_content") and msg_chunk.reasoning_content:
                #     yield {"type": "thought", "content": msg_chunk.reasoning_content}
                
                # # --- 原有的文本处理 ---
                # text = ""
                # if text: yield {"type": "text", "content": text}
                
                # 1. 解析文本内容 (Content)
                # 多模态支持：现在的模型不仅仅吐文本，还会吐图片、文件。
                # 所以 content 有时是 str，有时是 list[dict]（多模态块）。
                # 如果不判断，代码遇到图片块就会崩溃。
                content = getattr(msg_chunk, "content", "")
                text = ""
                if isinstance(content, str):
                    text = content
                elif isinstance(content, list):
                    text = "".join(
                        b.get("text", "") if isinstance(b, dict) else getattr(b, "text", "") 
                        for b in content
                    )
                
                if text:
                    yield {"type": "text", "content": text}

                # 2. 解析工具调用碎片 (Tool Calls)
                # 一个完整的工具调用名 get_weather 可能会被拆成 get_ 和 weather 两个碎片发过来。
                tool_chunks = getattr(msg_chunk, "tool_call_chunks", [])
                for tc in tool_chunks:
                    yield {
                        "type": "tool",
                        "name": tc.get("name"),
                        "args": tc.get("args")
                    }

In [16]:
print("--- Webster 正在思考 ---")

# 1. 启动原始流
raw_stream = agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="messages", 
    version="v2"
)

# 2. 使用处理器过滤并打印
for output in WebsterStreamProcessor.process(raw_stream):
    if output["type"] == "text":
        print(output["content"], end="", flush=True)
        
    elif output["type"] == "tool":
        if name := output["name"]:
            print(f"\n[Webster 决定使用工具: {name}]")
        if args := output["args"]:
            print(args, end="", flush=True)

print("\n--- 任务完成 ---")

--- Webster 正在思考 ---

[Webster 决定使用工具: get_weather]
{"city": "SF"}It's always sunny in SF!The weather in San Francisco is always sunny! 😊
--- 任务完成 ---


### LLM Tokens

要实时传输 LLM 生成的令牌，请使用`stream_mode="messages"`。您可以在下方看到代理流式传输工具调用的输出和最终响应。

**“微观世界”**：它不再关注整个对话的结果，而是让你实时观察 **每一个字符（Token）是从哪个“节点”蹦出来的**。

这也是为什么我们要学习 `metadata` 的原因——它给了你一个上帝视角，让你看到 Agent 在流式输出时的 **动态身份切换**。

#### 代码示例

##### 1. 核心代码拆解：你在观察什么？

这段代码的核心在于 `chunk["data"]` 的解包： `token, metadata = chunk["data"]`

- **`token` (AIMessageChunk)**：这是模型正在吐出的“碎片”。比如单词 "The" 或者工具调用的一个参数片段。
- **`metadata` (字典)**：这是碎片的“身份证”。其中最关键的字段是 **`langgraph_node`**。
  - 它告诉你这个碎片是 **`agent` 节点**（模型在思考/说话）产生的，还是 **`tools` 节点**（工具正在返回数据）产生的。

##### 2. 应用价值

在构建像 **Webster** 这样复杂的应用时，知道 `metadata['langgraph_node']` 有两个巨大的实战价值：

1. **精准过滤**：有时候你只想把 `agent` 节点说的话显示给用户，而不想让用户看到那些乱糟糟的 `tools` 节点的原始返回数据。通过判断 `metadata['langgraph_node'] == 'agent'`，你可以精准拦截。
2. **UI 状态切换**：
   - 当 `node` 是 `agent` 时，前端显示“Webster 正在回复...”。
   - 当 `node` 是 `tools` 时，前端显示“Webster 正在查阅天气插件...”。



In [17]:
from langchain.agents import create_agent


def get_weather(city: str) -> str:
    """Get weather for a given city."""

    return f"It's always sunny in {city}!"

model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)

agent = create_agent(
    model=model,
    tools=[get_weather],
)
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="messages",
    version="v2",
):
    if chunk["type"] == "messages":
        token, metadata = chunk["data"]
        print(f"node: {metadata['langgraph_node']}")
        print(f"content: {token.content_blocks}")
        print("\n")

node: model
content: [{'type': 'tool_call_chunk', 'id': 'e3c1079f-48c6-4b16-b0e6-2102ae706593', 'name': 'get_weather', 'args': '{"city": "SF"}'}]


node: model
content: []


node: model
content: []


node: tools
content: [{'type': 'text', 'text': "It's always sunny in SF!"}]


node: model
content: [{'type': 'text', 'text': 'The'}]


node: model
content: [{'type': 'text', 'text': ' weather'}]


node: model
content: [{'type': 'text', 'text': ' in'}]


node: model
content: [{'type': 'text', 'text': ' San'}]


node: model
content: [{'type': 'text', 'text': ' Francisco'}]


node: model
content: [{'type': 'text', 'text': ' is'}]


node: model
content: [{'type': 'text', 'text': ' always'}]


node: model
content: [{'type': 'text', 'text': ' sunny'}]


node: model
content: [{'type': 'text', 'text': '!'}]


node: model
content: [{'type': 'text', 'text': ' 😊'}]


node: model
content: []


node: model
content: []




#### 代码解析

通过这份结果，你可以看到 Agent 并不是一个静态的程序，而是一个在不同身份（Node）之间反复横跳的**生命体**。我们来逐行“破译”它传达的信息：

------

##### 1. 执行过程全解析

###### **第一阶段：决策与指令 (node: model)**

- **输出**：`'type': 'tool_call_chunk', 'name': 'get_weather', 'args': '{"city": "SF"}'`
- **解密**：这是模型（大脑）在发号施令。注意它叫 `tool_call_chunk`，说明在 `messages` 模式下，它是以“碎片”形式吐出来的。它告诉系统：“我需要去跑一下 `get_weather` 这个函数。”

###### **第二阶段：静默与切换 (node: model content: [])**

- **输出**：连续两个空列表 `[]`。
- **解密**：这非常关键！这是模型在**收尾**。它已经说完了指令，正在等待系统接管。这就像对讲机松开了按钮，发出了“Over”的信号。

###### **第三阶段：行动与观察 (node: tools)**

- **输出**：`'text': "It's always sunny in SF!"`
- **解密**：此时**权力交棒**到了 `tools` 节点。你的 Python 函数 `get_weather` 运行了。这里打印的是函数的返回值。
- **注意**：此时模型是“看不见”这段话的，它还在等系统把这个结果喂回给它。

###### **第四阶段：总结与回复 (node: model)**

- **输出**：`'text': 'The'`, `'text': ' weather'`...
- **解密**：系统把工具结果喂给了模型。模型第二次苏醒（第二次进入 `model` 节点），开始把学到的知识翻译成人话，逐字蹦出。

------

##### 2. 你发现了吗？两个有趣的细节：

1. **为什么会有这么多 `node: model`？** 因为在 LangChain 的逻辑里，`agent`（或 `model`）节点是核心控制器。它既负责“思考怎么做”，也负责“最后怎么说”。所以它在整个生命周期里会出现**两次**（甚至多次，如果需要多次调工具的话）。
2. **为什么会有 `content: []`？** 流式传输中，每个 Chunk 都要发送元数据。即使模型这一刻没吐字（比如它在处理内部逻辑或结束信号），它也会发一个空的 Chunk 告诉你：“我还在这个节点上，但现在没话写。”

------

##### 3. 这对开发 Webster 有什么启发？

如果你正在写前端 UI（比如一个网页聊天框）：

- **当你收到 `node: tools` 的消息时**：你可以在界面上显示一个搜索图标 🔍，或者提示“正在调用天气插件...”。
- **当你收到 `node: model` 且 content 为空时**：这是处理“等待状态”的最佳时机。
- **当 content 包含 `tool_call_chunk` 时**：你甚至可以实时展示 Agent 正在生成的参数，让用户觉得 Webster 特别智能，正在“精准计算”。

### Custom updates 自定义更新

要实时流式传输工具执行过程中的更新，可以使用get_stream_writer。

从**“被动接收者”**变成了**“主动广播者”**。

这体现了 LangGraph 中非常高级的一个意义：**Side-channel Communication（侧向通道通信）**。

#### 1. 核心意义：打破“黑盒”执行

在之前的 `updates` 或 `messages` 模式下，你是站在流水线外面观察。而这段代码里，你通过 `get_stream_writer()`，**从函数内部直接向外界“喊话”**。

- **打破阻塞**：通常工具调用（Tools）是一个黑盒。如果你的工具要去爬 10 个网页，或者计算 1 分钟，外界在 `updates` 模式下只能看到 `node: tools` 在运行，然后是漫长的死静。
- **实时进度条**：通过 `custom` 模式，你的工具可以在干活的过程中，源源不断地把“内部日志”吐出来（例如：正在扫描第 1 个文件... 正在解析数据...）。

> 意义在于**“交互的颗粒度”**。
>
> - **`messages`**：是模型（大脑）的流。
> - **`updates`**：是图（流程）的流。
> - **`custom`**：是**你（开发者）**定义的流。

#### 2. `stream_mode="custom"` 的独特之处

你会发现，这次输出**只有**你在 `writer()` 里写的那些话。

- **零噪音**：它过滤掉了所有的 `AIMessageChunk`（模型碎碎念）、`Metadata` 和 `Token`。
- **业务专注**：这个模式只传输你**显式**想要传输的内容。它不是为了给用户看“打字机效果”，而是为了传输**业务信号**。

#### 3. 在 Webster 里的实战价值

想象一下你的 Webster 以后要处理一个复杂的任务，比如“整理我的财务报表”：

| **模式**        | **用户看到的**                                               | **你的感受**                   |
| --------------- | ------------------------------------------------------------ | ------------------------------ |
| **普通模式**    | 屏幕卡住 30 秒，然后直接出结果。                             | “是不是死机了？”               |
| **Custom 模式** | 实时显示： 1. 正在读取 Excel... 2. 正在提取 2024 年数据... 3. 正在生成图表... | “它正在努力工作，很有安全感。” |

In [18]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer  


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"

model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)

agent = create_agent(
    model=model,
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="custom",
    version="v2",
):
    if chunk["type"] == "custom":
        print(chunk["data"])

Looking up data for city: SF
Acquired data for city: SF


In [19]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer


def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"

model = init_chat_model(
    model="qwen2.5:7b-instruct-q4_K_M", model_provider="ollama", temperature=0.1
)

agent = create_agent(
    model=model,
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["updates", "custom"],
    version="v2",
):
    print(f"stream_mode: {chunk['type']}")
    print(f"content: {chunk['data']}")
    print("\n")

stream_mode: updates
content: {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:7b-instruct-q4_K_M', 'created_at': '2026-04-10T09:12:58.3943488Z', 'done': True, 'done_reason': 'stop', 'total_duration': 5008111600, 'load_duration': 3548011200, 'prompt_eval_count': 157, 'prompt_eval_duration': 268840500, 'eval_count': 20, 'eval_duration': 1153971100, 'logprobs': None, 'model_name': 'qwen2.5:7b-instruct-q4_K_M', 'model_provider': 'ollama'}, id='lc_run--019d76aa-6b88-72e2-8925-6b0c8d765694-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'SF'}, 'id': 'db2dee68-fb81-461a-9f2c-e76d0577da0e', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 157, 'output_tokens': 20, 'total_tokens': 177})]}}


stream_mode: custom
content: Looking up data for city: SF


stream_mode: custom
content: Acquired data for city: SF


stream_mode: updates
content: {'tools': {'messages': [ToolMessage(content="It's always sunny in

In [20]:
def webster_stream_handler(stream):
    """优雅的流处理器：负责把原始数据转化为业务语言"""
    for chunk in stream:
        mode = chunk["type"]
        data = chunk["data"]

        if mode == "custom":
            # 自定义信号：直接返回字符串
            yield f"📢 [系统信号]: {data}"

        elif mode == "updates":
            # 节点更新：我们只关心哪个节点干了什么
            for node_name, content in data.items():
                yield f"✅ [节点完工]: {node_name}"
                # 如果有消息内容，尝试提取最后一条话
                if "messages" in content:
                    last_msg = content["messages"][-1]
                    if last_msg.content:
                        yield f"   └─ 内容预览: {last_msg.content[:30]}..."
                    if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
                        yield f"   └─ 决策: 准备调用工具 {last_msg.tool_calls[0]['name']}"


# --- 优雅的调用方式 ---
print("🚀 Webster 引擎启动...")
# 逻辑与展示完全分离
for message in webster_stream_handler(
    agent.stream(
        {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
        stream_mode=["updates", "custom"],
        version="v2",
    )
):
    print(message)

🚀 Webster 引擎启动...
✅ [节点完工]: model
   └─ 决策: 准备调用工具 get_weather
📢 [系统信号]: Looking up data for city: SF
📢 [系统信号]: Acquired data for city: SF
✅ [节点完工]: tools
   └─ 内容预览: It's always sunny in SF!...
✅ [节点完工]: model
   └─ 内容预览: The weather in San Francisco i...
